> 업로드 정리 주석(2026-10-30)
> - 이 노트북은 제공된 코드·해석의 기록이며 이번 정리에서 실제 데이터를 넣어 재실행하지 않았습니다.
> - 현재 코드의 분석 상한은 예측기준월 202602입니다. 아래 해석에 남은 2026.03·2026.02 원자료 등의 옛 수치는 현재 코드 재실행 결과로 확정하면 안 됩니다.
> - 전체 기간 Spearman 상관은 탐색 결과입니다. 최신 학습 구간 Pearson 분석 및 최종 변수 선택과 구분하세요.
> - 카드 13절의 라벨 분석은 202407~202508을 사용합니다. 최신 학습 범위(상관분석 기준 202301~202502)와 달라 모델 선택 근거로 재사용하기 전에 수정·재검증해야 합니다.
> - 낮은 상관만으로 독립성을 입증하거나 변수를 자동 채택하지 않습니다.
> - 실제 입력 데이터와 실행 출력은 공개 업로드 묶음에서 제외했습니다.


# 01. 카드 EDA
대상: `카드데이터_전처리_v05.csv` (v04 + 파생지표 8열). 계산 방법과 검증은 `카드_파생지표_결과.md` 참고.

**규칙**
- 원자료(`평균금액`, `평균건수`)는 **2026.01까지만** 본다. (목표: t = 2026.02 예측, 이후 자료는 쓰지 않음)
- 파생지표는 t = 2023.01 ~ 2026.02까지 본다. 2026.02 행이 시험 문제다.
- 폐업 라벨과 연결 짓는 분석은 하지 않는다. **단, 맨 끝 13번(혼동행렬) 섹션만 예외**이고, 시험 문제 정답은 쓰지 않도록 학습 구간(t = 2024.07~2025.08)만 쓴다.
- 그림마다 아래에 **"이상한 점과 이유"**를 한 줄씩 적는다. (맨 아래 정리 칸)

**구성** 1 불러오기 → 2 원자료 분포 → 3 객단가 → 4 시간 추이 → 5 업종·구 비교 → 6 녹산동 확인 → 7 파생지표 분포 → 8 지표 시간 추이·업종 비교 → 9 상관 → 10 극단값 → 11 점포 수 부족 조합 → 12 정리 → 13 지표별 혼동행렬

> ⚠ **시험 기준월을 2026.03 → 2026.02로 변경** (자료는 2026.01까지, 정답 기간 2026.02~2026.07). 위 설명과 코드(`CUT` 등)는 새 기준으로 고쳤다. **아래 '결과 해석'과 마지막 정리 표의 숫자·그림 설명은 이전 기준(자료 2026.02까지, 시험 2026.03)으로 쓴 것이라 한 달씩 어긋난다. 다시 실행해서 그림과 숫자를 새로 확인한 뒤 갱신한다.**


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "Malgun Gothic", "axes.unicode_minus": False,
                     "figure.dpi": 100, "axes.grid": True, "grid.alpha": .3})
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}".rstrip("0").rstrip("."))   # 8.3e+04 같은 지수 표기 대신 쉼표 숫자로 (소수는 최대 4자리)

V05 = "카드데이터_전처리_v05.csv"      # 이 노트북과 같은 폴더(완전쓰는거_ver2)에서 실행
CUT = 202601
IND = ["소비추세_6개월", "소비전년비", "소비변동계수_12개월", "점포당소비_z", "소비점포증가율격차"]

## 1. 불러오기와 첫 확인

In [ ]:
d = pd.read_csv(V05, encoding="utf-8-sig")
raw = d[d.기준월 <= CUT].copy()                         # 원자료 EDA는 2026.01까지
der = d[d.기준월 <= 202602].copy()                      # 파생지표는 2026.02(시험 문제)까지
ym = lambda s: pd.to_datetime(s.astype(str), format="%Y%m")
raw["월"] = ym(raw.기준월); der["월"] = ym(der.기준월)

print("v05 전체", d.shape, "| 원자료 사용", raw.shape, "| 파생지표 사용", der.shape)
print("기간", raw.기준월.min(), "~", raw.기준월.max(), "| 동", raw.행정동코드.nunique(), "| 구", raw.구군.nunique(), "| 업종", raw.카드업종.nunique())
print("키 중복", raw.duplicated(["기준월", "행정동코드", "카드업종"]).sum(), "| 원자료 빈칸", int(raw[["평균금액", "평균건수"]].isna().sum().sum()))
# 금액을 1.2e+09 같은 지수 표기 대신 쉼표 구분 숫자로 보여 준다
raw[["평균금액", "평균건수"]].describe().pipe(lambda d: d.apply(lambda s: s.map("{:,.0f}".format)))

### 결과 해석
- v05 전체 96,965행 중 **원자료는 2026.02까지 85,690행**, 파생지표용은 2026.03까지 87,945행을 쓴다. 동 205 · 구 16 · 업종 11, 키 중복 0, 빈칸 0.
- 평균금액 중앙값 약 1,790만 원, 최대 약 12억 원. 평균건수 중앙값 445건.
- **이상한 점:** 평균건수가 0인 행이 2건 있다(2025.03 `교육`, 중구 남포동·금정구 금성동). 두 곳은 원래 결제가 하루 1~2건뿐인 소액 칸이라, 그 달 일평균 건수가 0.5건 미만으로 떨어져 **반올림되어 0으로 보인 것**으로 판단한다(원자료의 반올림 방식은 직접 확인하지 못함). 객단가는 이 2행만 계산되지 않고, 지표에는 건수를 쓰지 않아 영향이 없다.


## 2. 원자료 분포 — 금액과 건수

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(np.log10(raw.평균금액), bins=60); ax[0].set_title("평균금액 (log10 원)")
ax[1].hist(np.log10(raw.평균건수.clip(lower=1)), bins=60); ax[1].set_title("평균건수 (log10 건, 0건은 1로)")
plt.tight_layout(); plt.show()
print("건수 0인 행", (raw.평균건수 == 0).sum(), "| 평균금액 분위수(원)")
# 금액을 1.2e+09 같은 지수 표기 대신 쉼표 구분 숫자로 보여 준다
raw.평균금액.quantile([0, .01, .25, .5, .75, .99, 1]).to_frame().T.pipe(lambda d: d.apply(lambda s: s.map("{:,.0f}".format)))

### 결과 해석
- **log10을 쓰는 이유:** 값이 약 4만 원 ~ 12억 원(3만 배 차)으로 퍼져 있어 그대로 그리면 대부분이 한쪽에 몰려 모양이 안 보인다. 로그를 씌우면 10배마다 눈금이 1씩 올라간다 (7 = 천만 원, 8 = 1억 원). 그림을 위한 변환일 뿐 원자료는 바꾸지 않았다.
- 금액: 대부분 6.5~8.5(약 300만 ~ 3억 원)에 있고 7.0~7.5 근처가 가장 많다. 업종마다 값 크기가 달라 봉우리가 여러 개로 보인다.
- 건수: 2~3(100~1,000건)에 몰려 있고 봉우리가 두 개(2.2, 3.0 근처)다. 왼쪽 끝의 작은 막대는 0건 행(로그를 구할 수 없어 1로 올려서 그림).
- **이상한 점:** 없음. 오른쪽으로 긴 분포라 로그 변환이 필요하다는 것을 확인했다.

## 3. 객단가 (평균금액 ÷ 평균건수)
금액과 건수를 한 번에 보는 값. 업종마다 다를 것이므로 업종별로 본다.

In [ ]:
raw["객단가"] = raw.평균금액 / raw.평균건수.replace(0, np.nan)
fig, ax = plt.subplots(figsize=(11, 4))
order = raw.groupby("카드업종").객단가.median().sort_values().index
ax.boxplot([raw.loc[raw.카드업종 == c, "객단가"].dropna() for c in order], showfliers=False)
ax.set_xticklabels(order)
ax.set_yscale("log"); ax.set_title("업종별 객단가 (원, log, 이상치 숨김)"); plt.xticks(rotation=30); plt.show()
raw.groupby("카드업종").객단가.describe()[["25%", "50%", "75%", "max"]].round(0).loc[order]

### 결과 해석
- 업종마다 객단가가 크게 다르다. 중앙값은 여행/숙박·음식료품 약 2.2만 원, 음식/주점 약 3.1만 원, 유통·여가/문화 약 3.3만 원, 생활·미용 약 5만 원, 의료 약 6.1만 원, 의류/잡화·자동차 약 6.2~6.5만 원, **교육 약 29.5만 원**.
- 자동차는 상자가 매우 넓다(약 1만 ~ 15만 원). 같은 업종 안 변동이 가장 크다.
- **해석:** 업종끼리 금액을 직접 비교하면 안 된다. 같은 업종 안에서 비교해야 한다(명세서 12번 z점수도 같은 업종 안에서 계산).

## 4. 시간 추이 — 전체 합계와 업종별 (2023.01 = 100)

In [ ]:
tot = raw.groupby("월")[["평균금액", "평균건수"]].sum()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
(tot.평균금액 / 1e8).plot(ax=ax[0]); ax[0].set_title("월별 일평균금액 합계 (억 원)")
(tot.평균건수 / 1e4).plot(ax=ax[1]); ax[1].set_title("월별 일평균건수 합계 (만 건)")
plt.tight_layout(); plt.show()

pv = raw.pivot_table(index="월", columns="카드업종", values="평균금액", aggfunc="sum")
(pv / pv.iloc[0] * 100).plot(figsize=(11, 4.5), title="업종별 월 합계 (2023.01 = 100)"); plt.legend(ncol=4, fontsize=8); plt.show()
print("전체 금액 합계 2023.01 → 2026.01 증가율", round(tot.평균금액.iloc[-1] / tot.평균금액.iloc[0] - 1, 3))
print("업종별 금액 증가율"); print((pv.iloc[-1] / pv.iloc[0] - 1).round(2).sort_values())

# 건수가 금액보다 더 오르는지 확인: 업종별 건수 지수, 건당 금액, 건수 증가율
cnt = raw.pivot_table(index="월", columns="카드업종", values="평균건수", aggfunc="sum")
(cnt / cnt.iloc[0] * 100).plot(figsize=(11, 4.5), title="업종별 월 건수 합계 (2023.01 = 100)"); plt.legend(ncol=4, fontsize=8); plt.show()
fig, ax = plt.subplots(figsize=(10, 3.2)); (tot.평균금액 / tot.평균건수).plot(ax=ax); ax.set_title("건당 금액 (원, 전체 합계)"); plt.show()
print("전체 건수 합계 2023.01 → 2026.01 증가율", round(tot.평균건수.iloc[-1] / tot.평균건수.iloc[0] - 1, 3))
print("건당 금액 2023.01 → 2026.01 증가율", round((tot.평균금액 / tot.평균건수).iloc[-1] / (tot.평균금액 / tot.평균건수).iloc[0] - 1, 3))
print("업종별 건수 증가율"); (cnt.iloc[-1] / cnt.iloc[0] - 1).round(2).sort_values()

### 결과 해석
- **월별 금액 합계**는 1,050~1,230억 사이에서 들쭉날쭉하게 움직이며 완만히 오른다. 최저 2023.03(1,049억), 최고 2025.09(1,232억). 2023.01 → 2026.02 증가율은 **+13.7%**.
- **건수 합계는 꾸준히 오른다**(+27.2%). 금액보다 더 많이 늘어서 **건당 금액이 43,503원 → 38,868원(−10.7%)으로 줄었다.** 소액 결제가 늘었다는 뜻이다.
- 월별 평균 금액 지수는 9월 1.055 · 11월 1.041 · 12월 1.053으로 높고 3월(0.954) · 8월(0.955)은 낮다. 계절성이 있다.
- **이상한 점 ①:** 2025.08 1,073억 → 2025.09 1,232억으로 **한 달에 +15%** 뛴다. 건수는 +3.3%뿐이라 건당 금액이 올랐다. 늘어난 금액은 유통 +60억, 생활 +45억, 의료 +24억 순이다. 원인은 확인하지 못했다.
- **이상한 점 ②:** 업종별 그림에서 **자동차가 2023.05에 201까지 튄다**(2023.02 147 → 04 162 → 05 201 → 06 156). 금액이 작은 업종이라 소액 변동이 크게 보인다. 교육과 자동차가 같은 파란색이라 그림만으로는 구분이 어렵다(교육의 최대는 2025.02의 116).
- 업종별 증가율: 음식료품 +51%, 의료 +33%, 미용 +28%, 음식/주점·의류/잡화 +19%, 교육 +15%, 여가/문화 +12%, 여행/숙박 +11%, 자동차 +10%, 유통 +7%, 생활 +1%. 업종마다 성장 속도가 다르다.

## 5. 업종·구 비교

In [ ]:
last = raw[raw.기준월 == CUT]
share = last.groupby("카드업종").평균금액.sum().sort_values()
gu = last.groupby("구군").평균금액.sum().sort_values()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
(share / share.sum()).plot.barh(ax=ax[0], title="업종 비중 (2026.01 금액)")
(gu / 1e8).plot.barh(ax=ax[1], title="구별 일평균금액 합계 (억 원, 2026.01)")
plt.tight_layout(); plt.show()
print("상위 5 동네(금액 합계)"); last.groupby(["구군", "읍면동"]).평균금액.sum().nlargest(5).round(0)

### 결과 해석
- **업종 비중(2026.02 금액):** 유통이 약 37%로 가장 크고, 음식/주점 16%, 생활 15%, 의료 12.5%, 음식료품 6% 순. 교육·여가/문화·여행/숙박은 각 약 3%, 자동차·의류/잡화·미용은 1.5% 안팎이다. (카드 '유통'의 세부 분류는 원자료 설명에 없어 확인하지 못했다.)
- **구별 합계:** 해운대구(약 158억)가 가장 크고 부산진구(131억), 동래구(106억), 남구(104억), 사하구(101억) 순. 중구(약 13억)가 가장 작다.
- 금액이 큰 동네: 기장군 정관읍, 해운대구 우3동, 남구 용호1동, 강서구 명지1동, 해운대구 재송1동. 신도시·주거 밀집 지역과 상권이 섞여 있다.
- **이상한 점:** 없음. 업종·구마다 규모가 크게 다르므로 구·업종 간 직접 비교는 피하고 변화율·순위를 쓰는 편이 안전하다.

## 6. 녹산동 확인 (신호동 합산: 2025.02 전후가 이어지는가)

In [ ]:
ns = raw[raw.읍면동 == "녹산동"].groupby("월").평균금액.sum() / 1e8
fig, ax = plt.subplots(figsize=(10, 3.2)); ns.plot(ax=ax, marker="o", ms=3)
ax.axvline(pd.Timestamp("2025-02-01"), color="r", ls="--"); ax.set_title("녹산동 월별 일평균금액 합계 (억 원, 빨간선 = 2025.02)")
plt.show()
ns.loc["2024-11":"2025-05"].round(2)

### 결과 해석
- 녹산동 월별 금액은 13~14.5억 사이에서 움직인다. 신호동을 합친 2025.02(빨간 선)에서 값이 끊기지 않는다(2025.01 14.1억 → 02 13.2억 → 03 13.3억). **합산이 잘 됐다.**
- 2025.02는 13.2억으로 낮은 편이지만, 합산 때문이 아니라 2024.12(14.6억)부터 이어진 하락이다.
- **이상한 점:** 2025.09~10에 14.3억 → 15.9억으로 올라 그래프 최고점이다. 같은 시기 전체 카드 금액의 점프(2025.09)와 겹친다.

## 7. 파생지표 분포 (t = 2023.01 ~ 2026.03)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(13, 6)); ax = ax.ravel()
for a, c in zip(ax, IND + ["점포수_t"]):
    v = der[c].dropna()
    a.hist(v.clip(v.quantile(.01), v.quantile(.99)), bins=60); a.set_title(c + " (1~99% 구간)")
plt.tight_layout(); plt.show()
der[IND + ["점포수_t"]].describe(percentiles=[.01, .05, .5, .95, .99]).T.round(3)

### 결과 해석
- **소비추세:** 0을 중심으로 좌우 대칭인 종 모양. 5%~95%가 −0.054 ~ 0.060이다. 양쪽 끝 막대는 1·99% 구간 밖을 잘라서 모은 것.
- **소비전년비:** 0 근처에 몰려 있고 오른쪽 꼬리가 길다. 최대 6.35(+635%).
- **소비변동계수:** 0.05~0.15에 몰려 있고 오른쪽 꼬리가 길다(0이 될 수 없는 값이라 한쪽만 긴 모양).
- **점포당소비_z:** 중심이 약 −0.3이고 오른쪽 꼬리가 길다. 비율 지표라 평균은 0이지만 중앙값은 음수다.
- **소비점포증가율격차:** 0을 중심으로 대칭. 5%~95%가 −0.18 ~ 0.24.
- **점포수_t:** 0~10곳 근처에 가장 많고(약 32,000행) 긴 오른쪽 꼬리. 중앙값 22곳.
- **이상한 점:** 소비전년비 > 2는 47건뿐이지만 소비추세 절댓값이 0.1을 넘는 행은 74,415행 중 2,587행(3.5%)이다. 극단값이 있으므로 모델에서 자르거나 순위로 바꾸는 것을 검토한다.

## 8. 지표의 시간 추이와 업종 비교

In [ ]:
med = der.groupby("월")[IND].median()
fig, ax = plt.subplots(2, 3, figsize=(13, 6)); ax = ax.ravel()
for a, c in zip(ax, IND):
    med[c].plot(ax=a); a.set_title(c + " (동×업종 중앙값)")
ax[-1].axis("off"); plt.tight_layout(); plt.show()

t = der[der.기준월 == 202602]
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, c in zip(ax, ["소비추세_6개월", "소비전년비", "소비변동계수_12개월"]):
    cs = t.groupby("카드업종")[c].median().sort_values().index
    a.boxplot([t.loc[t.카드업종 == k, c].dropna() for k in cs], showfliers=False)
    a.set_xticklabels(cs)
    a.set_title(c + " (2026.02)"); a.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()

### 결과 해석 (위: 지표의 시간 추이, 아래: 2026.03 업종별)
- **소비추세는 계절을 탄다.** 1월(2024.01, 2025.01, 2026.01)에 정점(+0.015~0.019), 7월 전후에 저점(−0.009)이다. 추세적 상승·하락은 없다.
- **소비전년비의 중앙값은 시간이 갈수록 올라간다.** 2024.07 −0.006 → 2025.01 0.015 → 2026.03 0.053. 카드 금액이 전반적으로 늘어난 영향으로, 학습 구간과 시험 구간(2026.03) 수준이 달라진다.
- 소비변동계수의 중앙값은 2024년에 0.098에서 0.084까지 내려갔다가 2025년에 0.09 근처로 오른다.
- 점포당소비_z의 중앙값은 완만히 내려간다(−0.255 → −0.270). 소비점포증가율격차는 −0.03 ~ +0.07 사이를 오르내리며 2026.03에 가장 높다(0.068).
- **업종별(2026.03):** 자동차는 소비추세가 가장 낮고(중앙값 −0.047) 상자가 가장 넓다. 소비전년비는 자동차 상자가 0.01~0.49로 매우 넓다. 소비변동계수는 음식/주점·유통이 약 0.04로 가장 안정적이고 자동차가 0.42로 가장 들쭉날쭉하다. 음식료품은 전년비가 높다(0.199).
- **이상한 점:** 시험 문제(2026.03)는 전년비·증가율 격차의 수준이 높은 쪽에 있다. 시간 순서로 나눠 학습할 때 주의가 필요하다(달별 순위 변환 검토).

## 9. 지표끼리 상관 (Spearman)

In [ ]:
cols = IND + ["점포수_t"]
cr = der[cols].corr(method="spearman")
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cr, vmin=-1, vmax=1, cmap="RdBu_r"); ax.grid(False)
ax.set_xticks(range(len(cols)), cols, rotation=45, ha="right"); ax.set_yticks(range(len(cols)), cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f"{cr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im); plt.tight_layout(); plt.show()
pairs = [(a, b, cr.loc[a, b]) for i, a in enumerate(cols) for b in cols[i+1:] if abs(cr.loc[a, b]) > 0.8]
print("상관 0.8 초과 쌍:", pairs or "없음")

### 결과 해석
- **상관이 0.8을 넘는 쌍이 없다.** 하나만 남겨야 할 지표는 없다.
- 가장 큰 상관은 소비전년비 × 소비점포증가율격차 0.46(격차 식에 소비 증가율이 들어 있어서)과 소비변동계수 × 점포수_t −0.55(점포가 적은 곳일수록 소비가 들쭉날쭉)다.
- **이상한 점:** 없음.

## 10. 극단값 (2026.03)

In [ ]:
show = ["구군", "읍면동", "카드업종", "평균금액", "점포수_t"]
for c, asc in [("소비추세_6개월", False), ("소비추세_6개월", True), ("소비전년비", False), ("점포당소비_z", False)]:
    print(f"== {c} {'하위' if asc else '상위'} 5")
    print(t.sort_values(c, ascending=asc).head(5)[show + [c]].to_string(index=False))

### 결과 해석 (2026.03)
- **소비추세 상·하위 대부분이 `자동차` 업종**이고 점포가 0~2곳이다(영도구 영선1동 +0.486, 중구 광복동 −0.345). 금액이 하루 16만~77만 원으로 작아 소액 변동이 크게 보인다.
- **소비전년비 최대**는 강서구 강동동의 자동차 +600%, 교육 +323%, 여행/숙박 +276%다. 강동동은 통신 지표에서도 극단값이다(03번 노트북 7번 참고).
- **점포당소비_z 상위**는 점포가 1곳뿐인 조합이다(서구 초장동 의료 9.1, 연제구 거제4동 여가/문화 8.5). 점포가 1곳이면 분모가 작아 값이 크게 튄다.
- **이상한 점:** 점포가 0곳인데 소비가 있는 조합이 있다(영도구 영선1동 자동차, 해운대구 우1동 자동차). 카드 업종과 인허가 업종의 범위가 다르기 때문이다(`쉬운설명_점포수0_원인.md`).

## 11. 점포가 적은 조합 (12·15번이 불안정한 곳)

In [ ]:
small = t.groupby("카드업종").apply(lambda g: pd.Series({"0곳": (g.점포수_t == 0).mean(), "5곳 미만": (g.점포수_t < 5).mean()}))
small.sort_values("5곳 미만").plot.barh(figsize=(8, 4), title="점포 0곳 / 5곳 미만 비율 (동×카드업종, 2026.02)"); plt.show()
print("점포 0곳", (t.점포수_t == 0).sum(), "/ 5곳 미만", (t.점포수_t < 5).sum(), "/ 전체", len(t))

### 결과 해석
- 2026.03 동×업종 2,255개 중 **점포 0곳이 151개, 5곳 미만이 574개**다.
- 업종별 5곳 미만 비율: **자동차 94%**, 의류/잡화 71%, 교육 56%, 여행/숙박 39%, 여가/문화 13%. 음식/주점·유통·음식료품은 0%에 가깝다.
- **이상한 점·해석:** 점포 0곳은 점포가 없어서가 아니라 인허가 목록에 그 종류의 가게가 거의 안 들어 있어서다(예: 의류/잡화는 안경점만). 이 4개 업종의 점포당소비_z·소비점포증가율격차는 믿기 어렵다.

## 12. 이상한 점과 이유 (정리)

각 셀 아래 '결과 해석'에 그림을 보고 읽은 내용을 적었다. 이 표는 핵심 수치만 모은 것이다.

| 항목 | 확인한 점 | 이유·해석 |
|---|---|---|
| 건수 0인 행 | 2행 (금액은 있음) | 객단가를 계산할 수 없어 빈칸 처리. 지표에는 평균건수를 쓰지 않아 영향 없음 |
| 금액 분포 | 중앙값 약 1,790만 원/일, 99% 4.0억, 최대 12억 | 오른쪽으로 매우 길다. 로그를 쓰는 이유 |
| 객단가 | 음식/주점 중앙값 약 3.1만 원 ~ 교육 약 29.5만 원, 자동차 최대 88만 원 | 업종마다 크게 다르므로 같은 업종 안에서만 비교 |
| 전체 증가율 | 2023.01 → 2026.02 금액 합계 +13.7% | 업종별로 음식료품 +51%, 의료 +33%, 미용 +28%, 생활 +1%. 소비전년비의 중앙값이 업종마다 다른 이유 |
| 녹산동 | 2025.01 14.1억 → 02 13.2억 → 03 13.3억 | 신호동 합산 후 끊김 없이 이어진다 |
| 상위 동네 | 기장군 정관읍, 해운대구 우3동, 남구 용호1동, 강서구 명지1동, 해운대구 재송1동 | 신도시·주거 밀집 지역과 상권이 섞여 있다 |
| 파생지표 상관 | 0.8 초과 쌍 없음 | 하나만 남겨야 할 지표 없음 |
| 극단값 | 상·하위 대부분 `자동차` 업종, 점포 0~2곳 | 소액 변동이 크게 보이고 점포가 적어 불안정 |
| 점포 0곳인데 소비가 있는 조합 | 영도구 영선1동 자동차(63만 원/일), 해운대구 우1동 자동차 등 | 카드 업종과 인허가 업종의 범위가 다르거나 인허가에 없는 점포 가능성. 점포당소비·격차는 빈칸 |
| 점포 부족 | 2026.03 동×업종 2,255개 중 0곳 151개, 5곳 미만 574개 | 점포당소비_z·증가율 격차가 불안정. 표본 보정 검토 |
| 소비전년비 극단 | 강서구 강동동 자동차 +600%, 교육 +323% | 소비가 작던 동네가 갑자기 커진 것. 원자료 확인 필요 |
| 지표별 혼동행렬 (13번) | 5개 지표 모두 AUC 0.47~0.51, 정밀도 3.4~5.8% (기본 폐업률 4.3%) | 카드 지표 하나만으로는 6개월 안 폐업을 거의 구분하지 못한다. 다른 지표와 함께 볼 때의 값은 모델링에서 본다 |

## 13. 지표별 혼동행렬 (폐업 라벨 연결, 이 섹션만 예외)

**질문:** 카드 파생지표 5개 각각으로 "6개월 안에 폐업할 점포"를 얼마나 가려내는가?

**방법**
- 단위: **점포 × 기준월**. t월 1일에 영업 중인 점포(개업월 < t ≤ 폐업월)마다, 그 점포의 동×카드업종 지표 값을 붙인다.
- 정답(폐업): 그 점포가 **t ~ t+5월 안에 폐업했으면 1**. 라벨은 `인허가_v2.1.xlsx`의 폐업년월이고, 같은 가게는 `점포이력ID`로 묶는다(카드_파생지표_결과.md와 같은 규칙).
- 기간: **t = 2024.07 ~ 2025.08**(14개월). 5개 지표가 모두 채워지는 첫 달부터, 6개월 결과가 2026.01 이전에 끝나는 마지막 달까지다. 시험 문제(2026.02~)의 정답은 쓰지 않는다.
- 분류 기준: **달별 위험 상위 20%**를 "위험"으로 표시한다. 위험 방향은 가설이다: 소비추세·소비전년비·점포당소비_z·소비점포증가율격차는 **작을수록**, 소비변동계수는 **클수록** 위험.
- 읽는 법: 아무 지표도 쓰지 않고 20%를 무작위로 뽑으면 **정밀도 = 기본 폐업률, 재현율 = 20%, AUC = 0.5**다. 이보다 얼마나 높은가를 본다.
- 시간이 걸린다(엑셀 읽기 약 1~2분).

In [ ]:
# 폐업 라벨 연결 (이 섹션만 예외). 점포 단위: t월에 영업 중인 점포가 t~t+5월(6개월) 안에 폐업했나
# 학습 구간 t = 2024.07~2025.08만 사용 → 6개월 결과가 2026.01 이전에 끝나서 시험 문제(2026.02~)의 정답은 쓰지 않는다
mi = lambda y: (y // 100) * 12 + y % 100                      # YYYYMM → 연속 월 번호
lb = pd.read_excel("인허가_v2.1.xlsx", sheet_name="점포", usecols=["점포ID", "점포이력ID", "카드업종", "행정동코드", "인허가일자", "인허가년월"]) \
       .merge(pd.read_excel("인허가_v2.1.xlsx", sheet_name="라벨")[["점포ID", "영업여부", "폐업년월"]], on="점포ID", how="left").sort_values("인허가일자")
gb = lb.groupby("점포이력ID")                                   # 같은 가게 = 점포이력ID (카드_파생지표_결과.md와 같은 규칙)
hist = pd.DataFrame({"개업": mi(gb.인허가년월.min()), "모두폐업": gb.영업여부.apply(lambda v: (v == "F").all()),
                     "폐업": gb.폐업년월.max(), "행정동코드": gb.행정동코드.last(), "카드업종": gb.카드업종.last()})
hist["폐업"] = np.where(hist.모두폐업, mi(hist.폐업.fillna(0).astype(int)), 10**9)   # 영업 중이면 아주 먼 미래
rows = []
for m in sorted(d.기준월.unique()):
    if not 202407 <= m <= 202508: continue
    k = mi(m); a = hist[(hist.개업 < k) & (hist.폐업 >= k)]     # t월 1일에 영업 중
    rows.append(pd.DataFrame({"기준월": m, "행정동코드": a.행정동코드.values, "카드업종": a.카드업종.values, "폐업": ((a.폐업 <= k + 5)).astype(int).values}))
stm = pd.concat(rows).merge(d[["기준월", "행정동코드", "카드업종"] + IND], on=["기준월", "행정동코드", "카드업종"], how="left")
print(f"점포×월 {len(stm):,}행 | 6개월 안에 폐업한 비율 {stm.폐업.mean():.3f} | 카드 지표가 안 붙은 행 {stm[IND].isna().any(axis=1).sum()}")

RISK = {"소비추세_6개월": -1, "소비전년비": -1, "소비변동계수_12개월": +1, "점포당소비_z": -1, "소비점포증가율격차": -1}   # +1 = 클수록 위험, −1 = 작을수록 위험 (가설)
def auc(y, x):                                                  # 위험 점수가 폐업을 얼마나 위에 세우나 (0.5 = 무작위)
    r = x.rank(); n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
res, mats = [], {}
for c, sg in RISK.items():
    z = stm.dropna(subset=[c]).copy(); z["risk"] = z[c] * sg
    z["표시"] = (z.risk >= z.groupby("기준월").risk.transform(lambda v: v.quantile(.8))).astype(int)   # 달별 위험 상위 20%
    tp, fp = ((z.표시 == 1) & (z.폐업 == 1)).sum(), ((z.표시 == 1) & (z.폐업 == 0)).sum()
    fn, tn = ((z.표시 == 0) & (z.폐업 == 1)).sum(), ((z.표시 == 0) & (z.폐업 == 0)).sum()
    mats[c] = np.array([[tn, fp], [fn, tp]])
    res.append({"지표": c, "정밀도": tp / (tp + fp), "재현율": tp / (tp + fn), "기본 폐업률": z.폐업.mean(), "AUC": auc(z.폐업, z.risk), "TP": tp, "FP": fp, "FN": fn, "TN": tn})
fig, axs = plt.subplots(1, 5, figsize=(17, 3.6))
for ax, (c, M) in zip(axs, mats.items()):
    ax.imshow(M / M.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2): ax.text(j, i, f"{M[i, j]:,}\n({M[i, j] / M[i].sum():.0%})", ha="center", va="center", fontsize=8)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["안전 표시", "위험 표시"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["유지", "폐업"]); ax.grid(False); ax.set_title(c, fontsize=9)
plt.suptitle("지표별 혼동행렬 (행 = 실제, 괄호 = 행 기준 비율, 달별 위험 상위 20% 표시)", y=1.04); plt.tight_layout(); plt.show()
pd.DataFrame(res).set_index("지표")


### 결과 해석
- 점포×월 **1,792,389행**(영업 중이던 점포 14개월 분), 6개월 안에 폐업한 비율은 **4.3%**(기본 폐업률). 카드 지표가 안 붙은 행은 101행뿐이다.
- 5개 지표 모두 **무작위와 거의 같다**: AUC 0.474~0.507, 정밀도 3.4~5.8%(기본 4.3%), 재현율 15.6~26.6%(무작위 20%).
- 그나마 나은 것은 **소비변동계수_12개월**: AUC 0.507, 정밀도 5.8%(기본의 약 1.3배), 재현율 26.6%.
- **소비전년비(0.479)·소비점포증가율격차(0.474)·점포당소비_z(0.485)·소비추세(0.494)는 0.5 미만**이다. 가설한 방향이 오히려 반대에 가깝거나 신호가 없다는 뜻이다. 반대 방향을 쓰면 소비전년비는 0.521이 된다. 결과를 보고 방향을 바꾸면 학습 구간에 맞추는 것이 되므로, 방향은 가설 그대로 두었다.
- **이유(추정, 확인하지는 않았다):** 카드 지표는 동×업종 단위라 같은 칸의 점포가 모두 같은 값을 받는다. 점포 고유 특성(업력, 프랜차이즈, 면적 등)이 없어서, 카드 지표 하나로는 같은 칸 안의 점포를 구분하지 못한다. 폐업률이 업종마다 다를 가능성이 있는데 업종별로 나누어 보지 않았다.
- **의미:** 이 결과는 "카드 지표가 쓸모없다"는 뜻이 아니다. 단일 지표 기준의 값이고, 업력·프랜차이즈 같은 점포 지표와 함께 쓸 때의 기여는 모델링에서 본다.
- **한계:** 라벨은 인허가 기준이고 학습 구간 14개월뿐이다. 점포당소비_z에는 점포 자신이 점포수에 포함되어 있다(자기 제외 미적용).